<a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MixedWM38 Wafer Defect Recognition — PyTorch Vision Transformer

這是原 TensorFlow/Keras notebook 的 **PyTorch / Google Colab** 版本。

- 資料集：MixedWM38 (`Wafer_Map_Datasets.npz`)
- 任務：8-bit 多標籤缺陷辨識（38 種標籤組合）
- 模型：自訂 Vision Transformer（13×13 patches）
- 損失：`BCEWithLogitsLoss`
- Colab：支援 GPU、AMP mixed precision、自動儲存最佳 checkpoint

> 在 Colab 請先選擇：**執行階段 → 變更執行階段類型 → T4 GPU**。

## 1. 安裝與匯入套件

In [ ]:
!pip -q install kagglehub

import os
import random
from pathlib import Path
from contextlib import nullcontext

import kagglehub
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 2. 下載並載入 MixedWM38

In [ ]:
dataset_dir = Path(kagglehub.dataset_download(
    'co1d7era/mixedtype-wafer-defect-datasets'
))

matches = list(dataset_dir.rglob('Wafer_Map_Datasets.npz'))
if not matches:
    raise FileNotFoundError(f'在 {dataset_dir} 中找不到 Wafer_Map_Datasets.npz')
dataset_path = matches[0]
print('Dataset:', dataset_path)

with np.load(dataset_path) as data:
    images = data['arr_0'].astype(np.float32)
    labels = data['arr_1'].astype(np.float32)

if images.ndim == 4 and images.shape[-1] == 1:
    images = images[..., 0]
if images.ndim != 3 or labels.ndim != 2 or labels.shape[1] != 8:
    raise ValueError(f'非預期資料形狀：images={images.shape}, labels={labels.shape}')

DEFECT_NAMES = ['Center', 'Donut', 'Edge_Loc', 'Edge_Ring',
                'Loc', 'Near_Full', 'Scratch', 'Random']

print(f'Images: {images.shape}, dtype={images.dtype}')
print(f'Labels: {labels.shape}, combinations={len(np.unique(labels, axis=0))}')

## 3. 切分資料與建立 DataLoader

In [ ]:
# 以 8-bit 標籤組合作為 stratify key，使 38 種組合在各 split 中比例接近。
label_keys = np.array([''.join(row.astype(int).astype(str)) for row in labels])
all_idx = np.arange(len(images))

train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=SEED, stratify=label_keys
)
train_keys = label_keys[train_idx]
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.10, random_state=SEED, stratify=train_keys
)

class WaferDataset(Dataset):
    def __init__(self, images, labels, indices, augment=False):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        x = torch.from_numpy(self.images[idx]).unsqueeze(0).float() / 2.0
        y = torch.from_numpy(self.labels[idx]).float()
        if self.augment:
            if torch.rand(()) < 0.5:
                x = torch.flip(x, dims=[2])
            # Wafer orientation is often arbitrary; 90-degree rotation is label preserving.
            x = torch.rot90(x, int(torch.randint(0, 4, ()).item()), dims=[1, 2])
        return x, y

BATCH_SIZE = 256  # 若 GPU 記憶體不足可改成 128 或 64
NUM_WORKERS = min(2, os.cpu_count() or 1)
pin_memory = device.type == 'cuda'

train_ds = WaferDataset(images, labels, train_idx, augment=True)
val_ds = WaferDataset(images, labels, val_idx)
test_ds = WaferDataset(images, labels, test_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin_memory,
                          persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=pin_memory,
                        persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=pin_memory,
                         persistent_workers=NUM_WORKERS > 0)

print(f'Train={len(train_ds):,}, Validation={len(val_ds):,}, Test={len(test_ds):,}')

## 4. 顯示 wafer 與 patches

In [ ]:
def read_label(label):
    names = [name for name, present in zip(DEFECT_NAMES, label) if present >= 0.5]
    return ', '.join(names) if names else 'Normal wafer'

def show_wafer(index):
    plt.figure(figsize=(4, 4))
    plt.imshow(images[index], cmap='viridis', vmin=0, vmax=2)
    plt.title(f'Wafer #{index}: {read_label(labels[index])}')
    plt.axis('off')
    plt.show()
    print('Label:', labels[index].astype(int))

show_wafer(int(train_idx[0]))

In [ ]:
PATCH_SIZE = 13
sample = torch.from_numpy(images[int(train_idx[0])]).unsqueeze(0).unsqueeze(0)
patches = nn.Unfold(kernel_size=PATCH_SIZE, stride=PATCH_SIZE)(sample)
patches = patches.squeeze(0).T.reshape(-1, PATCH_SIZE, PATCH_SIZE)

fig, axes = plt.subplots(1, len(patches), figsize=(12, 3))
for ax, patch in zip(axes, patches):
    ax.imshow(patch, cmap='viridis', vmin=0, vmax=2)
    ax.axis('off')
plt.suptitle(f'{len(patches)} non-overlapping {PATCH_SIZE}×{PATCH_SIZE} patches')
plt.tight_layout()
plt.show()

## 5. PyTorch Vision Transformer

`Conv2d(kernel_size=stride=13)` 等價於擷取不重疊 patches 後再做線性投影。輸出保留 logits；訓練時交給 `BCEWithLogitsLoss`，推論時才套 sigmoid。

In [ ]:
class WaferViT(nn.Module):
    def __init__(self, image_size=52, patch_size=13, in_channels=1,
                 num_labels=8, embed_dim=96, num_heads=4,
                 depth=16, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        if image_size % patch_size != 0:
            raise ValueError('image_size 必須可被 patch_size 整除')
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(in_channels, embed_dim,
                                     kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(self.num_patches * embed_dim, 2048),
            nn.GELU(), nn.Dropout(0.5),
            nn.Linear(2048, 1024),
            nn.GELU(), nn.Dropout(0.5),
            nn.Linear(1024, num_labels),
        )
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = self.pos_drop(x + self.pos_embed)
        x = self.encoder(x)
        return self.head(self.norm(x))

model = WaferViT().to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'Trainable parameters: {trainable:,}')

with torch.no_grad():
    xb, _ = next(iter(train_loader))
    print('Input:', tuple(xb.shape), 'Output:', tuple(model(xb[:2].to(device)).shape))

## 6. 訓練設定與訓練迴圈

In [ ]:
EPOCHS = 45       # 初次測試可先改為 3；正式訓練再改回 45
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
THRESHOLD = 0.5
CHECKPOINT_PATH = Path('/content/wafer_vit_best.pt')

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                              weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

def autocast_context():
    return torch.amp.autocast('cuda') if device.type == 'cuda' else nullcontext()

def run_epoch(model, loader, training=False):
    model.train(training)
    total_loss = total_bits = correct_bits = exact = 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        bar = tqdm(loader, leave=False)
        for x, y in bar:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                logits = model(x)
                loss = criterion(logits, y)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            pred = (torch.sigmoid(logits) >= THRESHOLD).float()
            total_loss += loss.item() * x.size(0)
            correct_bits += (pred == y).sum().item()
            total_bits += y.numel()
            exact += (pred == y).all(dim=1).sum().item()
            bar.set_postfix(loss=f'{loss.item():.4f}')
    return {
        'loss': total_loss / len(loader.dataset),
        'binary_acc': correct_bits / total_bits,
        'exact_acc': exact / len(loader.dataset),
    }

history = {'train_loss': [], 'val_loss': [], 'train_binary_acc': [],
           'val_binary_acc': [], 'train_exact_acc': [], 'val_exact_acc': []}
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    train_m = run_epoch(model, train_loader, training=True)
    val_m = run_epoch(model, val_loader, training=False)
    scheduler.step()
    for split, metrics in [('train', train_m), ('val', val_m)]:
        for key, value in metrics.items():
            history[f'{split}_{key}'].append(value)
    if val_m['loss'] < best_val_loss:
        best_val_loss = val_m['loss']
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch': epoch, 'best_val_loss': best_val_loss,
            'history': history, 'threshold': THRESHOLD,
        }, CHECKPOINT_PATH)
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss {train_m['loss']:.4f}, bit acc {train_m['binary_acc']:.4f}, exact {train_m['exact_acc']:.4f} | "
          f"val loss {val_m['loss']:.4f}, bit acc {val_m['binary_acc']:.4f}, exact {val_m['exact_acc']:.4f}")

print('Best checkpoint:', CHECKPOINT_PATH)

## 7. 訓練曲線

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs = range(1, len(history['train_loss']) + 1)
for ax, metric, title in zip(
    axes, ['loss', 'binary_acc', 'exact_acc'],
    ['Loss', 'Per-label binary accuracy', 'Exact-match accuracy']
):
    ax.plot(epochs, history[f'train_{metric}'], label='Train')
    ax.plot(epochs, history[f'val_{metric}'], label='Validation')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

## 8. 載入最佳模型並測試

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded epoch {checkpoint['epoch']}, val loss={checkpoint['best_val_loss']:.4f}")

model.eval()
all_probs, all_targets = [], []
with torch.no_grad():
    for x, y in tqdm(test_loader):
        with autocast_context():
            logits = model(x.to(device, non_blocking=True))
        all_probs.append(torch.sigmoid(logits).cpu())
        all_targets.append(y)

probs = torch.cat(all_probs).numpy()
targets = torch.cat(all_targets).numpy().astype(int)
preds = (probs >= THRESHOLD).astype(int)

print(f'Binary accuracy: {(preds == targets).mean():.4f}')
print(f'Exact-match accuracy: {(preds == targets).all(axis=1).mean():.4f}')
print(f'Micro F1: {f1_score(targets, preds, average="micro", zero_division=0):.4f}')
print(f'Macro F1: {f1_score(targets, preds, average="macro", zero_division=0):.4f}')
print('\nPer-label report:')
print(classification_report(targets, preds, target_names=DEFECT_NAMES, zero_division=0))

## 9. 單張 wafer 預測

In [ ]:
def predict_one(test_position=0):
    x, target = test_ds[test_position]
    model.eval()
    with torch.no_grad(), autocast_context():
        probability = torch.sigmoid(model(x.unsqueeze(0).to(device))).cpu()[0]
    prediction = (probability >= THRESHOLD).int().numpy()

    plt.figure(figsize=(4, 4))
    plt.imshow(x.squeeze().numpy() * 2, cmap='viridis', vmin=0, vmax=2)
    plt.title(f'Prediction: {read_label(prediction)}')
    plt.axis('off'); plt.show()
    print('Ground truth:', target.int().numpy(), '→', read_label(target.numpy()))
    print('Prediction  :', prediction, '→', read_label(prediction))
    print('Probability :', np.round(probability.numpy(), 3))

predict_one(0)

## 10. 下載 checkpoint（選用）

In [ ]:
# 在 Colab 執行以下兩行即可下載最佳權重：
# from google.colab import files
# files.download(str(CHECKPOINT_PATH))